In [1]:
!pip install networkx plotly pyvis


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyvis.network as net
from IPython.display import display, HTML

## 1. Загрузка и первичный анализ данных

In [3]:
# Список названий столбцов, соответствующих структуре данных
columns = [
    'label',                            # 0: метка (F... для мошеннических)
    'sender_user_id',                   # 1: ID отправителя (пользователь)
    'receiver_user_id',                 # 2: ID получателя
    'sender_account_id',                # 3: ID счета отправителя
    'receiver_account_id',              # 4: ID счета получателя
    'amount',                           # 5: сумма транзакции
    'tx_type',                          # 6: тип транзакции (Ind, Dt, Wl, Merchant, ArRC)
    'status',                           # 7: статус (SU - успешно, FA - неуспешно)
    'sender_bal_before',                # 8: баланс отправителя до
    'sender_bal_after',                 # 9: баланс отправителя после
    'receiver_bal_before',              # 10: баланс получателя до
    'receiver_bal_after',               # 11: баланс получателя после
    'unused1', 'unused2',               # 12-13: не используются
    'unused3', 'unused4',               # 14-15: не используются
    'timestamp_sender',                 # 16: временная метка (отправитель)
    'timestamp_receiver',               # 17: временная метка (получатель)
    'sender_account_id2',               # 18: дубль ID счета отправителя
    'unused5', 'unused6',               # 19-20: пустые
    'unused7',                          # 21: не используются
    'transaction_time',                 # 22: время транзакции (совпадает с 15,16)
    'sender_type',                      # 23: тип отправителя (EU, Mer, RET, ...)
    'receiver_type'                     # 24: тип получателя
]

# Загрузка данных из CSV с разделителем '|'
df = pd.read_csv('FinFraud_Labelled.csv', sep='|', header=None, names=columns, dtype=str)

print("Размер данных:", df.shape)
print("Первые 5 строк:")
display(df.head())

Размер данных: (54848, 25)
Первые 5 строк:


,label,sender_user_id,receiver_user_id,sender_account_id,receiver_account_id,amount,tx_type,status,sender_bal_before,sender_bal_after,...,unused4,timestamp_sender,timestamp_receiver,sender_account_id2,unused5,unused6,unused7,transaction_time,sender_type,receiver_type
0,N-RegC2C,PN_EU_3_4,PN_EU_0_883,EUAcc3_4,EUAcc0_883,68897.74,Ind,SU,100000000.00,99930413.28,...,NaN,01/06/2011 00:09:01,01/06/2011 00:09:01,EUAcc3_4,NaN,NaN,C2C201161.099,01/06/2011 00:09:01,EU,EU
1,N-RegC2C,PN_EU_1_139,PN_EU_0_754,EUAcc1_139,EUAcc0_754,68945.47,Ind,SU,100000000.00,99930365.08,...,NaN,01/06/2011 00:15:23,01/06/2011 00:15:23,EUAcc1_139,NaN,NaN,C2C201161.01515,01/06/2011 00:15:23,EU,EU
2,N-RegDep,PN_Ret2,PN_EU_3_17,RAcc2,EUAcc3_17,9715.41,Dt,SU,1000000000.00,999990284.59,...,NaN,01/06/2011 00:22:07,01/06/2011 00:22:07,RAcc2,NaN,NaN,Dp201161.02222,01/06/2011 00:22:07,RET,EU
3,N-RegDep,PN_Ret1,PN_EU_0_266,RAcc1,EUAcc0_266,79303.74,Dt,SU,1000000000.00,999920696.26,...,NaN,01/06/2011 00:22:35,01/06/2011 00:22:35,RAcc1,NaN,NaN,Dp201161.02222,01/06/2011 00:22:35,RET,EU
4,N_Reg_RC,PN_EU_0_905,operator,EUAcc0_905,A0,929.92,ArRC,SU,100000000.00,99999070.08,...,NaN,01/06/2011 00:29:56,01/06/2011 00:29:56,EUAcc0_905,NaN,NaN,Rc201161.02929,01/06/2011 00:29:56,EU,operator


In [4]:
# Числовые столбцы приводим к float
numeric_cols = ['amount', 'sender_bal_before', 'sender_bal_after', 'receiver_bal_before', 'receiver_bal_after']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Временные столбцы приводим к datetime
time_cols = ['timestamp_sender', 'timestamp_receiver', 'transaction_time']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Флаг мошенничества: метка начинается с 'F'
df['is_fraud'] = df['label'].str.startswith('F', na=False)

print(
    f"Статистика по флагу мошенничества:\n{df['is_fraud'].value_counts()}",
    f"Типы транзакций:\n{df['tx_type'].value_counts()}",
    f"Типы отправителя:\n{df['sender_type'].value_counts()}",
    f"Типы получателя:\n{df['receiver_type'].value_counts()}",
    sep='\n\n'
)

Статистика по флагу мошенничества:
is_fraud
False    53170
True      1678
Name: count, dtype: int64

Типы транзакций:
tx_type
ArRC        28312
Dt          12867
Ind          8205
Wl           5021
Merchant      443
Name: count, dtype: int64

Типы отправителя:
sender_type
EU     41981
RET    12867
Name: count, dtype: int64

Типы получателя:
receiver_type
operator    28312
EU          21072
RET          5021
MER           443
Name: count, dtype: int64


In [5]:
# Разделение данных на мошеннические и нормальные
fraud_df = df[df['is_fraud'] == True]
normal_df = df[df['is_fraud'] == False]

print(
    f"Всего транзакций: {len(df)}",
    f"Мошеннических: {len(fraud_df)}",
    f"Нормальных: {len(normal_df)}",
    sep='\n'
)

Всего транзакций: 54848
Мошеннических: 1678
Нормальных: 53170


## 2. Построение графа транзакций

In [6]:
# Функция построения графа транзакций
def build_graph(dataframe, edge_attrs=None):
    G = nx.DiGraph()
    for idx, row in dataframe.iterrows():
        u = row['sender_user_id']
        v = row['receiver_user_id']
        if pd.isna(u) or pd.isna(v):
            continue
        # Добавляем узлы с атрибутами типа
        if not G.has_node(u):
            G.add_node(u, type=row['sender_type'])
        if not G.has_node(v):
            G.add_node(v, type=row['receiver_type'])
        # Добавляем ребро
        attrs = {
            'amount': row['amount'],
            'tx_type': row['tx_type'],
            'timestamp': row['transaction_time'],
            'is_fraud': row['is_fraud']
        }
        if edge_attrs:
            attrs.update(edge_attrs)
        G.add_edge(u, v, **attrs)

    # Вычисляем степень узлов и масштабируем размер для визуализации
    degrees = dict(G.degree())
    max_deg = max(degrees.values()) if degrees else 1
    min_deg = min(degrees.values()) if degrees else 0
    for node, deg in degrees.items():
        size = 5 + 35 * (deg - min_deg) / (max_deg - min_deg) if max_deg > min_deg else 10
        G.nodes[node]['size'] = size
    return G

In [7]:
# Полный граф на всех данных
G_full = build_graph(df)
print(f"Построен граф для визуализации: {G_full.number_of_nodes()} узлов, {G_full.number_of_edges()} рёбер")

# Мошеннический подграф
G_fraud = build_graph(fraud_df)
fraud_nodes = set()
for u, v in G_fraud.edges():
    fraud_nodes.add(u)
    fraud_nodes.add(v)
# Ограничиваемся только узлами, участвующими в мошеннических транзакциях
G_fraud = G_full.subgraph(fraud_nodes)
print(f"Построен подграф для визуализации: {G_fraud.number_of_nodes()} узлов, {G_fraud.number_of_edges()} рёбер")

# Расширенный мошеннический подграф: включаем всех соседей мошеннических узлов
subgraph_nodes = set(fraud_nodes)
for node in fraud_nodes:
    for neighbor in G_full.neighbors(node):
        subgraph_nodes.add(neighbor)
G_sub = G_full.subgraph(subgraph_nodes)
print(f"Построен подграф для визуализации: {G_sub.number_of_nodes()} узлов, {G_sub.number_of_edges()} рёбер")

Построен граф для визуализации: 2009 узлов, 7476 рёбер
Построен подграф для визуализации: 108 узлов, 537 рёбер
Построен подграф для визуализации: 921 узлов, 4962 рёбер


## 3. Визуализация с помощью Pyvis (интерактивный HTML)

In [8]:
# Функция создания интерактивного графа с помощью Pyvis
def create_pyvis_network(G, graph_title="full", height="800px", width="100%", node_color="#424242", fraud_edge_color="red", normal_edge_color="gray"):
    nt = net.Network(height=height, width=width, directed=True)
    nt.set_options("""
    var options = {
        "physics": {
            "enabled": true,
            "solver": "forceAtlas2Based",
            "forceAtlas2Based": {
                "gravitationalConstant": -10,
                "centralGravity": 0.01,
                "springLength": 100
            }
        }
    }
    """)
    
    # Добавляем узлы с цветом в зависимости от типа
    for node in G.nodes():
        node_type = G.nodes[node].get('type', 'unknown')
        if node_type == 'EU':
            color = "#0000ff"
        elif node_type == 'Mer':
            color = "#ff0000"
        elif node_type == 'RET':
            color = "#00ff00"
        else:
            color = node_color
        size = G.nodes[node].get('size', 10)  # значение по умолчанию 10
        nt.add_node(node, label=str(node), title=f"ID: {node}<br>Тип: {node_type}", color=color, size=size)
    
    # Добавляем рёбра (мошеннические выделяем красным)
    for u, v, data in G.edges(data=True):
        is_fraud = data.get('is_fraud', False)
        color = fraud_edge_color if is_fraud else normal_edge_color
        title = f"Сумма: {data.get('amount', 'N/A')}<br>Тип: {data.get('tx_type', 'N/A')}<br>Время: {data.get('timestamp', 'N/A')}"
        nt.add_edge(u, v, color=color, title=title, width=1 if is_fraud else 0.1)
    
    # Сохраняем
    nt.save_graph(f"pyvis_{graph_title}_graph.html")
    return nt

In [9]:
print("Создание интерактивных графов с Pyvis...")

nt = create_pyvis_network(G_fraud, graph_title="fraud")
nt = create_pyvis_network(G_sub, graph_title="sub")
nt = create_pyvis_network(G_full)

display(HTML("Файлы .html созданы. Откройте их в браузере для интерактивного просмотра."))

Создание интерактивных графов с Pyvis...


## 4. Визуализация с помощью Plotly (статический граф)

In [10]:
# Функция построения статического графа с Plotly
def plotly_graph(G, title="Граф транзакций", fraud_edges_only=False):
    pos = nx.spring_layout(G, k=1, iterations=50, seed=42)
    
    # Координаты и атрибуты узлов
    node_x = []
    node_y = []
    node_text = []
    node_color = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(f"ID: {node}<br>Тип: {G.nodes[node].get('type', 'unknown')}")
        node_type = G.nodes[node].get('type', 'unknown')
        if node_type == 'EU':
            node_color.append("#0000ff")
        elif node_type == 'Mer':
            node_color.append("#ff0000")
        elif node_type == 'RET':
            node_color.append("#00ff00")
        else:
            node_color.append("#424242")

    # Разделяем рёбра на мошеннические и нормальные
    fraud_edges = [(u, v, d) for u, v, d in G.edges(data=True) if d.get('is_fraud', False)]
    normal_edges = [(u, v, d) for u, v, d in G.edges(data=True) if not d.get('is_fraud', False)]
    
    edge_traces = []
    for edges, color, width, name in [(fraud_edges, 'red', 1, 'Fraud'), (normal_edges, 'gray', 0.2, 'Normal')]:
        if not edges:
            continue
        edge_x = []
        edge_y = []
        for u, v, _ in edges:
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            edge_x.append(x0)
            edge_x.append(x1)
            edge_x.append(None)
            edge_y.append(y0)
            edge_y.append(y1)
            edge_y.append(None)
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=width, color=color),
            hoverinfo='none',
            mode='lines',
            name=name
        )
        edge_traces.append(edge_trace)
    
    # Узлы
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        text=node_text,
        marker=dict(
            size=[G.nodes[node].get('size', 10) for node in G.nodes()],
            color=node_color,
            line=dict(width=1, color='black')
        ),
        name='Nodes'
    )
    
    # Собираем фигуру
    fig = go.Figure(data=edge_traces + [node_trace],
                    layout=go.Layout(
                        title=title,
                        showlegend=True,
                        hovermode='closest',
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        plot_bgcolor='rgba(0,0,0,0)'
                    ))
    return fig

In [11]:
fraud_graph_fig = plotly_graph(G_fraud, title="Подграф с мошенническими транзакциями")
fraud_graph_fig.show()

In [12]:
sub_graph_fig = plotly_graph(G_sub, title="Подграф с мошенническими транзакциями (расширенный)")
sub_graph_fig.show()

In [13]:
full_graph_fig = plotly_graph(G_full, title="Полный граф транзакций")
full_graph_fig.show()

## 5. Анализ мошеннических схем

### Схема 1: Кража телефона
Характерные признаки:
- Резкое изменение баланса (списание крупной суммы с жертвы)
- Транзакция типа Wl (снятие) или Merchant (покупка) сразу после кражи
- Участие агента (RET) или торговца (Mer)
- Необычное время (например, ночью)

In [14]:
fraud_edges = list(G_fraud.edges(data=True))
print("Мошеннические рёбра (первые 5):")
for u, v, d in fraud_edges[:5]:
    print(f"{u} -> {v}, сумма: {d['amount']}, тип: {d['tx_type']}")

Мошеннические рёбра (первые 5):
PN_EU_0_672 -> PN_Ret6, сумма: 5197.2, тип: Wl
PN_EU_0_672 -> PN_Ret5, сумма: 18137.4, тип: Wl
PN_EU_0_672 -> PN_Ret4, сумма: 18400.97, тип: Wl
PN_EU_0_1135 -> PN_Ret3, сумма: 5245.19, тип: Wl
PN_EU_0_1135 -> PN_Ret5, сумма: 41534.03, тип: Wl


In [15]:
fraud_types = fraud_df['tx_type'].value_counts()
print("Типы мошеннических транзакций:")
print(fraud_types)

Типы мошеннических транзакций:
tx_type
Wl     957
Ind    721
Name: count, dtype: int64


In [16]:
fraud_amounts = fraud_df['amount'].describe()
print("Статистика сумм мошеннических транзакций:")
print(fraud_amounts)

Статистика сумм мошеннических транзакций:
count     1678.000000
mean     10979.051794
std       7570.244308
min         11.350000
25%       4926.657500
50%       9874.605000
75%      15709.565000
max      50864.490000
Name: amount, dtype: float64


In [17]:
victim_candidates = fraud_df.groupby('sender_user_id')['amount'].sum().sort_values(ascending=False)
print("Кандидаты в жертвы (по общей сумме списаний):")
print(victim_candidates.head(10))

Кандидаты в жертвы (по общей сумме списаний):
sender_user_id
PN_EU_0_955     2018313.06
PN_EU_1_328     1920590.23
PN_EU_0_260     1901076.94
PN_EU_0_1045    1800401.56
PN_EU_0_5        372153.87
PN_EU_1_449      257517.81
PN_EU_0_1113     247823.73
PN_EU_1_437      246587.22
PN_EU_0_668      244329.03
PN_EU_0_248      240088.86
Name: amount, dtype: float64


### Схема 2: Ботнет-атака
Характерные признаки:
- Много мелких транзакций от множества отправителей к одному получателю (или наоборот)
- Высокая степень исходящих/входящих связей у некоторых узлов

In [18]:
fraud_degrees = dict(G_fraud.degree())
top_degree_nodes = sorted(fraud_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
print("Узлы с наибольшей степенью в мошенническом подграфе (возможные центры ботнета):")
for node, deg in top_degree_nodes:
    print(f"{node}: степень {deg}")

Узлы с наибольшей степенью в мошенническом подграфе (возможные центры ботнета):
PN_Ret4: степень 67
PN_Ret5: степень 64
PN_Ret6: степень 64
PN_Ret3: степень 63
PN_Ret2: степень 60
PN_Ret1: степень 57
PN_EU_1_328: степень 48
PN_EU_0_260: степень 47
PN_EU_0_955: степень 45
PN_EU_0_1045: степень 44


In [19]:
hub = top_degree_nodes[0][0]
hub_edges = list(G_fraud.in_edges(hub, data=True)) + list(G_fraud.out_edges(hub, data=True))
print(f"Транзакции узла {hub}:")
for u, v, d in hub_edges[:10]:
    print(f"{u} -> {v}, сумма: {d['amount']}, тип: {d['tx_type']}")

Транзакции узла PN_Ret4:
PN_EU_3_36 -> PN_Ret4, сумма: 343224.85, тип: Wl
PN_EU_1_328 -> PN_Ret4, сумма: 93.11, тип: Wl
PN_EU_0_25 -> PN_Ret4, сумма: 17765.43, тип: Wl
PN_EU_0_176 -> PN_Ret4, сумма: 14062.3, тип: Wl
PN_EU_0_373 -> PN_Ret4, сумма: 7973.74, тип: Wl
PN_EU_0_624 -> PN_Ret4, сумма: 33451.78, тип: Wl
PN_EU_0_935 -> PN_Ret4, сумма: 18146.95, тип: Wl
PN_EU_0_763 -> PN_Ret4, сумма: 17723.72, тип: Wl
PN_EU_1_49 -> PN_Ret4, сумма: 15722.62, тип: Wl
PN_EU_0_441 -> PN_Ret4, сумма: 9682.59, тип: Wl


## 6. Построение графических паттернов

In [20]:
# Функция построения эго-графа (подграф, содержащий узел и его окрестность)
def ego_graph(G, node, radius=2):
    nodes = {node}
    for _ in range(radius):
        neighbors = set()
        for n in nodes:
            neighbors.update(G.neighbors(n))
        nodes.update(neighbors)
    return G.subgraph(nodes)

In [21]:
victim = victim_candidates.index[0]
print(f"Выбрана жертва: {victim}")
print(f"Общая сумма списаний (отправленных средств): {victim_candidates[victim]}")

G_victim = ego_graph(G_fraud, victim, radius=2)
fig_victim = plotly_graph(G_victim, title=f"Подграф жертвы {victim} (кража телефона)")
fig_victim.show()

Выбрана жертва: PN_EU_0_955
Общая сумма списаний (отправленных средств): 2018313.06


In [22]:
hub = top_degree_nodes[0][0]
hub_degree = top_degree_nodes[0][1]
print(f"Выбран центр ботнета: {hub} (степень = {hub_degree})")

G_hub = ego_graph(G_fraud, hub, radius=2)
fig_hub = plotly_graph(G_hub, title=f"Подграф центра ботнета {hub}")
fig_hub.show()

Выбран центр ботнета: PN_Ret4 (степень = 67)


## 7. Формулирование признаков мошенничества

На основе построенных графических иллюстраций и количественного анализа можно выделить следующие характерные признаки:

1. Кража мобильного телефона (схема «жертва -> агент»)
   - Отправитель – обычный пользователь (EU), у которого резко уменьшается баланс.
   - Получатель – агент (RET) или торговец (Mer).
   - Тип транзакции – Wl (снятие) или Merchant (покупка).
   - Сумма – крупная, часто превышающая средний размер обычных переводов.
   - Временная метка – может указывать на нетипичное время (например, ночь).
   - Графовая структура – изолированная пара или небольшая цепочка, где жертва соединена с одним-двумя получателями.
2. Ботнет-атака (схема «множество пользователей -> агент»)
   - Узел-агент (RET) имеет аномально высокую степень (много входящих рёбер).
   - Отправители – большое количество обычных пользователей (EU).
   - Тип транзакций – преимущественно Wl или Ind (переводы между пользователями).
   - Суммы – могут быть как мелкими (распыление), так и крупными (сбор средств).
   - Графовая структура – звездообразная: агент в центре, множество листьев-отправителей.